In [1]:
import ee
import pandas as pd

In [ ]:
aoi = ee.Geometry.Rectangle([138.35, -35.30, 139.10, -34.50])

season1_start, season1_end = '2021-01-01', '2021-03-31'   
season2_start, season2_end = '2021-04-01', '2021-06-30'   
season3_start, season3_end = '2021-07-01', '2021-09-30'   
season4_start, season4_end = '2021-10-01', '2021-12-31'   

bands_10m = ['B2', 'B3', 'B4', 'B8']
bands_20m = ['B5', 'B6', 'B7', 'B8A', 'B11', 'B12']
all_bands = bands_10m + bands_20m

scale = 10
MAX_CLOUD_PERCENTAGE = 20
CLOUD_THRESHOLD = 0.6

In [3]:
# =============================================================================
# Prepare Cloud-Masked Sentinel-2 Collections
# =============================================================================

QA_BAND = "cs_cdf"


def apply_cloud_mask(image):
    """
    Apply Cloud Score+ mask and convert DN to reflectance.
    """
    return (
        image
        .updateMask(image.select(QA_BAND).gte(CLOUD_THRESHOLD))
        .select(all_bands)
        .divide(10000)
        .copyProperties(image, image.propertyNames())
    )


def get_clean_collection(start_date, end_date):
    """
    Returns a cloud-masked Sentinel-2 ImageCollection.
    """

    collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(aoi)
        .filterDate(start_date, end_date)
        .filter(
            ee.Filter.lt(
                "CLOUDY_PIXEL_PERCENTAGE",
                MAX_CLOUD_PERCENTAGE
            )
        )
        .linkCollection(
            ee.ImageCollection(
                "GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED"
            ),
            [QA_BAND]
        )
        .map(apply_cloud_mask)
    )

    return collection


# -----------------------------------------------------------------------------
# Seasonal Collections
# -----------------------------------------------------------------------------

season1_collection = get_clean_collection(
    season1_start,
    season1_end
)

season2_collection = get_clean_collection(
    season2_start,
    season2_end
)

season3_collection = get_clean_collection(
    season3_start,
    season3_end
)

season4_collection = get_clean_collection(
    season4_start,
    season4_end
)

In [4]:
# =============================================================================
# Create Seasonal Median Composites
# =============================================================================

def create_composite(collection, start_date):
    """
    Creates a cloud-free seasonal median composite.
    """

    return (
        collection
        .median()
        .clip(aoi)
        .set("system:time_start", ee.Date(start_date).millis())
    )


# -----------------------------------------------------------------------------
# Seasonal Median Composites
# -----------------------------------------------------------------------------

season1_composite = create_composite(
    season1_collection,
    season1_start
)

season2_composite = create_composite(
    season2_collection,
    season2_start
)

season3_composite = create_composite(
    season3_collection,
    season3_start
)

season4_composite = create_composite(
    season4_collection,
    season4_start
)

In [5]:
# =============================================================================
# Rename Seasonal Bands
# =============================================================================

def rename_bands(image, season_prefix):
    """
    Rename bands by adding a seasonal prefix.
    Example:
        B2  -> Q1_B2
        B3  -> Q1_B3
    """

    renamed_bands = [
        f"{season_prefix}_{band}"
        for band in all_bands
    ]

    return image.rename(renamed_bands)


# -----------------------------------------------------------------------------
# Rename Bands
# -----------------------------------------------------------------------------

season1_composite = rename_bands(
    season1_composite,
    "Q1"
)

season2_composite = rename_bands(
    season2_composite,
    "Q2"
)

season3_composite = rename_bands(
    season3_composite,
    "Q3"
)

season4_composite = rename_bands(
    season4_composite,
    "Q4"
)

In [6]:
# =============================================================================
# Merge Seasonal Composites
# =============================================================================

final_image = (
    season1_composite
    .addBands(season2_composite)
    .addBands(season3_composite)
    .addBands(season4_composite)
)

In [7]:
# =============================================================================
# Load ESA WorldCover Land Cover Labels
# =============================================================================

worldcover = (
    ee.ImageCollection("ESA/WorldCover/v200")
    .first()
    .select("Map")
    .clip(aoi)
)

In [8]:
# =============================================================================
# Stratified Sampling
# =============================================================================

SAMPLES_PER_CLASS = 10000

training_image = final_image.addBands(worldcover.rename("label"))

samples = training_image.stratifiedSample(
    numPoints=SAMPLES_PER_CLASS,
    classBand="label",
    region=aoi,
    scale=10,
    seed=42,
    geometries=True,
    dropNulls=True

)

In [9]:
# print(samples.aggregate_histogram("label").getInfo())

In [10]:
# # =============================================================================
# # Export Training Dataset to Google Drive
# # =============================================================================

export_task = ee.batch.Export.table.toDrive(
    collection=samples,
    description="lulc_training_dataset_2.0",
    folder="",
    fileNamePrefix="lulc_training_dataset_2.0",
    fileFormat="CSV"
)

export_task.start()

print("Export Started Successfully!")

In [11]:
# =============================================================================
# Export Full Seasonal Feature Image for Model Prediction
# =============================================================================

export_image_task = ee.batch.Export.image.toDrive(
    image=final_image,
    description="lulc_full_prediction_image",
    folder="",
    fileNamePrefix="lulc_full_prediction_image",
    region=aoi,
    scale=10,
    fileFormat="GeoTIFF",
    maxPixels=1e13
)

export_image_task.start()

print("Full prediction image export started successfully!")

Full prediction image export started successfully!
